# Notebook 04 — Machine Learning Churn Prediction

## Loading Dataset

In this section, we will load the cleaned customer churn dataset created during the data preparation stage.

The purpose is to:
- Import the required Python libraries
- Load the final customer dataset
- Confirm that the dataset is available for machine learning analysis

In [1]:
# Import required libraries

import pandas as pd
import numpy as np


# Load customer churn dataset

file_path = "/kaggle/input/datasets/neelamsingh1987/customer-churn-dataset/customer_churn_final.csv"

df = pd.read_csv(file_path)


# Display first 5 rows of the dataset

df.head()

,customer_id,signup_date,region,age_group,subscription_type,monthly_revenue,tenure_months,monthly_login_count,average_session_time_minutes,features_used,last_login_days_ago,complaint_count,response_time_hours,resolution_time_hours,customer_satisfaction_score,churn
0,10001,2023-01-31,London,60+,Professional,79,35,19.666667,28.124427,5.083333,29.666667,0.0,0.000000,0.000000,10.0,0
1,10002,2023-12-30,Leeds,18-25,Professional,79,24,14.583333,24.804342,5.833333,34.166667,2.0,13.014601,56.085322,9.0,0
2,10003,2022-05-10,Bristol,26-35,Basic,29,44,14.333333,24.192618,4.916667,30.250000,1.0,4.425014,6.192709,7.0,0
3,10004,2023-07-18,London,18-25,Basic,29,29,17.083333,22.674429,5.416667,26.583333,0.0,0.000000,0.000000,10.0,0
4,10005,2023-02-04,Manchester,60+,Basic,29,35,10.333333,24.052414,4.500000,28.083333,3.0,22.489549,82.144367,9.0,0


## Dataset Structure

Before building a machine learning model, we need to understand the structure of the dataset.

We will check:
- Number of rows and columns
- Available features
- Data types

This helps confirm that the dataset is suitable for machine learning.

In [2]:
# Check dataset size

df.shape

(10000, 16)

## Checking Available Features

Machine learning models require clear input variables.

In this step, we will review all available columns and identify:
- Customer information features
- Behaviour features
- Support features
- The target variable (`churn`)

In [3]:
# Display all column names

df.columns

Index(['customer_id', 'signup_date', 'region', 'age_group',
       'subscription_type', 'monthly_revenue', 'tenure_months',
       'monthly_login_count', 'average_session_time_minutes', 'features_used',
       'last_login_days_ago', 'complaint_count', 'response_time_hours',
       'resolution_time_hours', 'customer_satisfaction_score', 'churn'],
      dtype='object')

## Check Data Types

Before building a machine learning model, we need to understand the data types of each column.

Machine learning algorithms require numerical inputs, so we need to identify:
- Numerical variables
- Categorical variables
- Date variables

This will help determine the preprocessing steps required before model training.

In [4]:
# Display data types for each column

df.dtypes

customer_id                       int64
signup_date                      object
region                           object
age_group                        object
subscription_type                object
monthly_revenue                   int64
tenure_months                     int64
monthly_login_count             float64
average_session_time_minutes    float64
features_used                   float64
last_login_days_ago             float64
complaint_count                 float64
response_time_hours             float64
resolution_time_hours           float64
customer_satisfaction_score     float64
churn                             int64
dtype: object

## Check Missing Values

We will check whether any missing values exist in the dataset.

Missing values can affect machine learning performance, so they need to be identified before model training.

In [5]:
# Check missing values in each column

df.isnull().sum()

customer_id                     0
signup_date                     0
region                          0
age_group                       0
subscription_type               0
monthly_revenue                 0
tenure_months                   0
monthly_login_count             0
average_session_time_minutes    0
features_used                   0
last_login_days_ago             0
complaint_count                 0
response_time_hours             0
resolution_time_hours           0
customer_satisfaction_score     0
churn                           0
dtype: int64

## Separate Features and Target

The target variable is the outcome we want the machine learning model to predict.

In this project:
- The target variable is churn
- All other relevant customer information will be used as predictive features

We will separate the dataset into:
- X: input features
- y: target variable

In [6]:
# Separate features and target variable

X = df.drop("churn", axis=1)

y = df["churn"]


# Display the shape of features and target

print("Features shape:", X.shape)

print("Target shape:", y.shape)


Features shape: (10000, 15)
Target shape: (10000,)


## Check Target Distribution

Before training a classification model, we need to check the distribution of the target variable.

This shows how many customers:
- Stayed with the company
- Churned

This is important because an imbalanced target can affect model performance.

In [7]:
# Check churn distribution

y.value_counts()

churn
0    8496
1    1504
Name: count, dtype: int64

## Removing Unnecessary Columns

Some columns are not useful for machine learning prediction.

The customer ID is only an identifier and does not provide any behavioural information.

The signup date is stored as text and will not be used in the initial model.

These columns will be removed before preparing the dataset for machine learning.

In [8]:
# Remove identifier and date columns

X = X.drop(["customer_id", "signup_date"], axis=1)


# Display remaining columns

X.head()

,region,age_group,subscription_type,monthly_revenue,tenure_months,monthly_login_count,average_session_time_minutes,features_used,last_login_days_ago,complaint_count,response_time_hours,resolution_time_hours,customer_satisfaction_score
0,London,60+,Professional,79,35,19.666667,28.124427,5.083333,29.666667,0.0,0.000000,0.000000,10.0
1,Leeds,18-25,Professional,79,24,14.583333,24.804342,5.833333,34.166667,2.0,13.014601,56.085322,9.0
2,Bristol,26-35,Basic,29,44,14.333333,24.192618,4.916667,30.250000,1.0,4.425014,6.192709,7.0
3,London,18-25,Basic,29,29,17.083333,22.674429,5.416667,26.583333,0.0,0.000000,0.000000,10.0
4,Manchester,60+,Basic,29,35,10.333333,24.052414,4.500000,28.083333,3.0,22.489549,82.144367,9.0


## Encoding Categorical Variables

Machine learning algorithms require numerical inputs.

The dataset contains categorical variables such as region, age group, and subscription type.

We will convert these categories into numerical columns using one-hot encoding.

This allows the machine learning model to use these variables during training.

In [9]:
# Import encoding function

from sklearn.preprocessing import OneHotEncoder


# Identify categorical columns

categorical_columns = [
    "region",
    "age_group",
    "subscription_type"
]


# Apply one-hot encoding

X_encoded = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True
)


# Display new dataset structure

X_encoded.head()

,monthly_revenue,tenure_months,monthly_login_count,average_session_time_minutes,features_used,last_login_days_ago,complaint_count,response_time_hours,resolution_time_hours,customer_satisfaction_score,...,region_Glasgow,region_Leeds,region_London,region_Manchester,age_group_26-35,age_group_36-45,age_group_46-60,age_group_60+,subscription_type_Enterprise,subscription_type_Professional
0,79,35,19.666667,28.124427,5.083333,29.666667,0.0,0.000000,0.000000,10.0,...,False,False,True,False,False,False,False,True,False,True
1,79,24,14.583333,24.804342,5.833333,34.166667,2.0,13.014601,56.085322,9.0,...,False,True,False,False,False,False,False,False,False,True
2,29,44,14.333333,24.192618,4.916667,30.250000,1.0,4.425014,6.192709,7.0,...,False,False,False,False,True,False,False,False,False,False
3,29,29,17.083333,22.674429,5.416667,26.583333,0.0,0.000000,0.000000,10.0,...,False,False,True,False,False,False,False,False,False,False
4,29,35,10.333333,24.052414,4.500000,28.083333,3.0,22.489549,82.144367,9.0,...,False,False,False,True,False,False,False,True,False,False


## Checking Final Feature Dataset

After encoding categorical variables, we will check the final structure of the feature dataset.

This confirms:
- The number of features available for modelling
- That all variables are now machine learning compatible

In [10]:
# Check final feature dataset shape

X_encoded.shape

(10000, 21)

## Splitting Data into Training and Testing Sets

Before training the machine learning model, the dataset is divided into two parts.

The training dataset is used to learn patterns from customer behaviour.

The testing dataset is used to evaluate how well the model predicts unseen customers.

An 80/20 split will be used while maintaining the original churn distribution.

In [11]:
# Import train-test split function

from sklearn.model_selection import train_test_split


# Split dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# Check shapes

print("X_train:", X_train.shape)

print("X_test:", X_test.shape)

print("y_train:", y_train.shape)

print("y_test:", y_test.shape)

X_train: (8000, 21)
X_test: (2000, 21)
y_train: (8000,)
y_test: (2000,)


## Checking Churn Distribution After Data Split

The training and testing datasets should maintain the same churn distribution as the original dataset.

This prevents the model evaluation from being affected by an uneven class distribution.

In [12]:
# Check churn distribution in training and testing data

print("Training churn distribution:")
print(y_train.value_counts(normalize=True))


print("\nTesting churn distribution:")
print(y_test.value_counts(normalize=True))

Training churn distribution:
churn
0    0.849625
1    0.150375
Name: proportion, dtype: float64

Testing churn distribution:
churn
0    0.8495
1    0.1505
Name: proportion, dtype: float64


## Logistic Regression Model

Logistic Regression is used as the first baseline classification model.

The model predicts the probability that a customer will churn based on their characteristics.

The output will be evaluated using:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC score

In [13]:
# Import Logistic Regression model

from sklearn.linear_model import LogisticRegression


# Create model

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)


# Train the model

logistic_model.fit(
    X_train,
    y_train
)


# Display confirmation

print("Logistic Regression model trained successfully")

Logistic Regression model trained successfully


## Making Predictions

After training the model, we use the testing dataset to generate predictions.

The model will predict whether each customer is likely to churn.

These predictions will then be compared with the actual churn values to evaluate performance.

In [14]:
# Generate predictions on test data

y_pred = logistic_model.predict(X_test)


# Display first 10 predictions

y_pred[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

## Evaluating Logistic Regression Performance

The model performance will be measured using classification evaluation metrics.

These metrics help us understand how accurately the model predicts customer churn.

Because identifying potential churners is the main business objective, recall will be an important metric.

In [15]:
# Import evaluation metrics

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


# Calculate evaluation metrics

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)


# ROC-AUC requires prediction probabilities

y_prob = logistic_model.predict_proba(X_test)[:,1]

roc_auc = roc_auc_score(y_test, y_prob)


# Display results

print("Accuracy:", accuracy)

print("Precision:", precision)

print("Recall:", recall)

print("F1 Score:", f1)

print("ROC-AUC:", roc_auc)

Accuracy: 0.9385
Precision: 0.8588709677419355
Recall: 0.707641196013289
F1 Score: 0.7759562841530054
ROC-AUC: 0.9736272460446735


## Logistic Regression Model Summary

The first machine learning model was developed using Logistic Regression as a baseline classification approach.

The model was trained using 80% of the customer dataset and evaluated on 20% unseen customer data.

The model performance was measured using accuracy, precision, recall, F1-score, and ROC-AUC.

Model performance:

- Accuracy: 93.85%
- Precision: 85.89%
- Recall: 70.76%
- F1-score: 77.60%
- ROC-AUC: 97.36%

Business interpretation:

The Logistic Regression model provides a strong baseline for predicting customer churn.

The model correctly identifies a high proportion of customers, achieving strong accuracy and ROC-AUC performance.

The recall score of 70.76% indicates that the model can identify approximately 7 out of 10 customers who are likely to churn.

However, some churn-risk customers may still be missed, which is important because losing customers creates revenue impact.

This baseline model will be compared with more advanced machine learning approaches to determine whether prediction performance can be improved.

## Random Forest Model

Random Forest is an ensemble machine learning algorithm that combines multiple decision trees to make predictions.

Compared with Logistic Regression, Random Forest can identify more complex patterns in customer behaviour.

The model will be trained and evaluated using the same training and testing datasets to allow a fair comparison.

In [16]:
# Import Random Forest classifier

from sklearn.ensemble import RandomForestClassifier


# Create Random Forest model

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)


# Train the model

rf_model.fit(
    X_train,
    y_train
)


# Display confirmation

print("Random Forest model trained successfully")

Random Forest model trained successfully


## Making Predictions

After training the Random Forest model, predictions will be generated using the unseen testing dataset.

These predictions will be compared with the actual churn values to evaluate model performance.

In [17]:
# Generate predictions using Random Forest

y_pred_rf = rf_model.predict(X_test)


# Display first 10 predictions

y_pred_rf[:10]

array([0, 0, 0, 0, 0, 0, 0, 1, 0, 0])

## Evaluating Random Forest Performance

The Random Forest model will be evaluated using the same classification metrics as the Logistic Regression model.

The evaluation metrics include:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

Using the same metrics allows a fair comparison between different machine learning models.

In [18]:
# Calculate Random Forest evaluation metrics

accuracy_rf = accuracy_score(y_test, y_pred_rf)

precision_rf = precision_score(y_test, y_pred_rf)

recall_rf = recall_score(y_test, y_pred_rf)

f1_rf = f1_score(y_test, y_pred_rf)


# Calculate ROC-AUC using prediction probabilities

y_prob_rf = rf_model.predict_proba(X_test)[:,1]

roc_auc_rf = roc_auc_score(y_test, y_prob_rf)


# Display results

print("Accuracy:", accuracy_rf)

print("Precision:", precision_rf)

print("Recall:", recall_rf)

print("F1 Score:", f1_rf)

print("ROC-AUC:", roc_auc_rf)

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
ROC-AUC: 1.0


## Random Forest Model Evaluation Summary

The Random Forest model achieved perfect performance on the testing dataset.

Performance results:

- Accuracy: 100%
- Precision: 100%
- Recall: 100%
- F1-score: 100%
- ROC-AUC: 100%

Although these results indicate excellent predictive ability, perfect classification performance is uncommon in real-world churn problems.

Further investigation is required to check for possible data leakage or overly strong predictive features before selecting the final model.

## Random Forest Feature Importance

Random Forest provides feature importance scores that show which variables contribute most to prediction.

Analysing feature importance helps identify the main factors driving churn predictions and can reveal potential data leakage.

In [19]:
# Create feature importance dataframe

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})


# Sort features by importance

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)


# Display top features

feature_importance.head(10)

,Feature,Importance
4,features_used,0.251289
2,monthly_login_count,0.229130
5,last_login_days_ago,0.183059
6,complaint_count,0.162343
9,customer_satisfaction_score,0.132212
7,response_time_hours,0.010879
8,resolution_time_hours,0.009946
3,average_session_time_minutes,0.006948
1,tenure_months,0.005687
0,monthly_revenue,0.001010


## Random Forest Feature Importance Summary

Feature importance analysis shows which variables contributed most to predicting customer churn.

The strongest predictive features were:

- Features used (25.13%)
- Monthly login count (22.91%)
- Last login days ago (18.31%)
- Complaint count (16.23%)
- Customer satisfaction score (13.22%)

Business interpretation:

The model indicates that customer engagement and experience factors are the strongest indicators of churn risk.

Customers with lower platform usage, reduced login activity, more complaints, and lower satisfaction levels appear more likely to churn.

These insights can help the business design targeted retention strategies focused on improving engagement and customer experience.

## Random Forest Confusion Matrix

A confusion matrix shows the number of correct and incorrect predictions made by the model.

It helps identify:

- True negatives: Customers correctly predicted as retained
- True positives: Customers correctly predicted as churned
- False negatives: Customers who churned but were missed by the model
- False positives: Customers predicted as churned but actually retained

This provides a deeper understanding of model performance beyond overall accuracy.

In [20]:
# Import confusion matrix

from sklearn.metrics import confusion_matrix


# Create confusion matrix

cm_rf = confusion_matrix(y_test, y_pred_rf)


# Display confusion matrix

cm_rf

array([[1699,    0],
       [   0,  301]])

## Random Forest Confusion Matrix Summary

The confusion matrix shows that the Random Forest model correctly classified all customers in the testing dataset.

Results:

- Correctly identified retained customers: 1,699
- Incorrectly classified retained customers: 0
- Correctly identified churned customers: 301
- Missed churned customers: 0

Business interpretation:

The model successfully identified all churned customers within the test dataset.

However, perfect classification performance is uncommon in real-world customer churn problems.

Further validation and model comparison are required before selecting the final predictive model.

## Gradient Boosting Model

Gradient Boosting is an ensemble machine learning algorithm that combines multiple decision trees to improve prediction performance.

The model learns from previous prediction errors and gradually improves classification accuracy.

The model will be trained using the same training data and evaluated using the same performance metrics as previous models.

In [21]:
# Import Gradient Boosting classifier

from sklearn.ensemble import GradientBoostingClassifier


# Create Gradient Boosting model

gb_model = GradientBoostingClassifier(
    random_state=42
)


# Train the model

gb_model.fit(
    X_train,
    y_train
)


# Display confirmation

print("Gradient Boosting model trained successfully")

Gradient Boosting model trained successfully


## Making Predictions

After training the Gradient Boosting model, predictions will be generated using the unseen testing dataset.

These predictions will be compared with the actual churn values to evaluate the model performance.

In [22]:
# Generate predictions using Gradient Boosting

y_pred_gb = gb_model.predict(X_test)


# Display first 10 predictions

y_pred_gb[:10]

array([0, 0, 0, 0, 0, 0, 0, 1, 0, 0])

## Evaluating Gradient Boosting Performance

The Gradient Boosting model will be evaluated using the same classification metrics used for previous models.

The evaluation metrics include:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

Using consistent evaluation metrics allows comparison between all machine learning models.

In [23]:
# Calculate Gradient Boosting evaluation metrics

accuracy_gb = accuracy_score(y_test, y_pred_gb)

precision_gb = precision_score(y_test, y_pred_gb)

recall_gb = recall_score(y_test, y_pred_gb)

f1_gb = f1_score(y_test, y_pred_gb)


# Calculate ROC-AUC using prediction probabilities

y_prob_gb = gb_model.predict_proba(X_test)[:,1]

roc_auc_gb = roc_auc_score(y_test, y_prob_gb)


# Display results

print("Accuracy:", accuracy_gb)

print("Precision:", precision_gb)

print("Recall:", recall_gb)

print("F1 Score:", f1_gb)

print("ROC-AUC:", roc_auc_gb)

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0
ROC-AUC: 1.0


## Comparing Machine Learning Models

Three classification models were developed to predict customer churn:

- Logistic Regression
- Random Forest
- Gradient Boosting

The models were evaluated using accuracy, precision, recall, F1-score, and ROC-AUC.

Comparing multiple models helps identify the most suitable approach for predicting customer churn.

In [24]:
# Create model comparison dataframe

model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Accuracy": [
        accuracy,
        accuracy_rf,
        accuracy_gb
    ],
    "Precision": [
        precision,
        precision_rf,
        precision_gb
    ],
    "Recall": [
        recall,
        recall_rf,
        recall_gb
    ],
    "F1 Score": [
        f1,
        f1_rf,
        f1_gb
    ],
    "ROC-AUC": [
        roc_auc,
        roc_auc_rf,
        roc_auc_gb
    ]
})


# Display comparison

model_comparison

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.9385,0.858871,0.707641,0.775956,0.973627
1,Random Forest,1.0000,1.000000,1.000000,1.000000,1.000000
2,Gradient Boosting,1.0000,1.000000,1.000000,1.000000,1.000000


## Machine Learning Model Comparison Summary

Three machine learning models were developed to predict customer churn:

- Logistic Regression
- Random Forest
- Gradient Boosting

The models were evaluated using accuracy, precision, recall, F1-score, and ROC-AUC.

Model comparison results:

| Model | Accuracy | Precision | Recall | F1 Score | ROC-AUC |
|---|---|---|---|---|---|
| Logistic Regression | 93.85% | 85.89% | 70.76% | 77.60% | 97.36% |
| Random Forest | 100% | 100% | 100% | 100% | 100% |
| Gradient Boosting | 100% | 100% | 100% | 100% | 100% |

Business interpretation:

Logistic Regression provided a strong baseline model with good predictive performance and high ROC-AUC.

Random Forest and Gradient Boosting achieved perfect performance on the testing dataset.

Because perfect classification is uncommon in real-world churn prediction problems, additional validation should be performed before final model selection.

Further analysis will focus on:
- Model validation
- Understanding customer risk factors
- Generating churn probability scores
- Creating customer risk segments for retention strategies

## Cross-Validation Performance Check

Cross-validation is used to test whether the machine learning models perform consistently across different subsets of the dataset.

Instead of evaluating the model using only one train-test split, the dataset is divided into multiple folds.

This provides a more reliable estimate of model performance and helps identify potential overfitting.

In [25]:
# Import cross-validation function

from sklearn.model_selection import cross_val_score


# Perform 5-fold cross-validation for Logistic Regression

logistic_cv = cross_val_score(
    logistic_model,
    X_encoded,
    y,
    cv=5,
    scoring="accuracy"
)


# Perform 5-fold cross-validation for Random Forest

rf_cv = cross_val_score(
    rf_model,
    X_encoded,
    y,
    cv=5,
    scoring="accuracy"
)


# Perform 5-fold cross-validation for Gradient Boosting

gb_cv = cross_val_score(
    gb_model,
    X_encoded,
    y,
    cv=5,
    scoring="accuracy"
)


# Display individual fold results

print("Logistic Regression CV Scores:")
print(logistic_cv)

print("\nRandom Forest CV Scores:")
print(rf_cv)

print("\nGradient Boosting CV Scores:")
print(gb_cv)

Logistic Regression CV Scores:
[0.9555 0.951  0.94   0.953  0.9505]

Random Forest CV Scores:
[1.     0.9995 1.     1.     1.    ]

Gradient Boosting CV Scores:
[0.9995 0.9995 1.     1.     1.    ]


## Average Cross-Validation Performance

The average cross-validation score provides a single measure of model consistency across all folds.

This helps compare the reliability of each machine learning model.

In [26]:
# Calculate average cross-validation accuracy

cv_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Average CV Accuracy": [
        logistic_cv.mean(),
        rf_cv.mean(),
        gb_cv.mean()
    ]
})


# Display results

cv_results

,Model,Average CV Accuracy
0,Logistic Regression,0.9500
1,Random Forest,0.9999
2,Gradient Boosting,0.9998


## Cross-Validation Summary

Cross-validation was performed to evaluate model consistency across multiple data splits.

Average cross-validation accuracy:

| Model | Average CV Accuracy |
|---|---|
| Logistic Regression | 95.00% |
| Random Forest | 99.99% |
| Gradient Boosting | 99.98% |

Business interpretation:

The validation results confirm that the ensemble models (Random Forest and Gradient Boosting) consistently outperform the Logistic Regression baseline.

Random Forest achieved the highest average cross-validation accuracy and will be selected as the final prediction model.

However, the near-perfect performance should be interpreted carefully, as the dataset contains strong behavioural patterns associated with churn.

The final model will now be used to generate customer churn risk predictions and identify customers requiring retention actions.

## Generating Customer Churn Risk Scores

The Random Forest model can estimate the probability that each customer will churn.

These probabilities allow customers to be ranked by risk level.

Higher churn probabilities indicate customers who may require immediate retention action.

In [27]:
# Generate churn probability scores

churn_probability = rf_model.predict_proba(X_encoded)[:,1]


# Create customer risk dataframe

customer_risk = X_encoded.copy()


# Add churn probability

customer_risk["churn_probability"] = churn_probability


# Display first rows

customer_risk.head()

,monthly_revenue,tenure_months,monthly_login_count,average_session_time_minutes,features_used,last_login_days_ago,complaint_count,response_time_hours,resolution_time_hours,customer_satisfaction_score,...,region_Leeds,region_London,region_Manchester,age_group_26-35,age_group_36-45,age_group_46-60,age_group_60+,subscription_type_Enterprise,subscription_type_Professional,churn_probability
0,79,35,19.666667,28.124427,5.083333,29.666667,0.0,0.000000,0.000000,10.0,...,False,True,False,False,False,False,True,False,True,0.00
1,79,24,14.583333,24.804342,5.833333,34.166667,2.0,13.014601,56.085322,9.0,...,True,False,False,False,False,False,False,False,True,0.00
2,29,44,14.333333,24.192618,4.916667,30.250000,1.0,4.425014,6.192709,7.0,...,False,False,False,True,False,False,False,False,False,0.00
3,29,29,17.083333,22.674429,5.416667,26.583333,0.0,0.000000,0.000000,10.0,...,False,True,False,False,False,False,False,False,False,0.00
4,29,35,10.333333,24.052414,4.500000,28.083333,3.0,22.489549,82.144367,9.0,...,False,False,True,False,False,False,True,False,False,0.01


## Customer Risk Segmentation

Churn probability scores are converted into customer risk categories.

This allows the business to prioritise retention activities.

Risk categories:

- High Risk: Customers with high probability of churn
- Medium Risk: Customers requiring engagement monitoring
- Low Risk: Customers with low immediate churn risk

In [28]:
# Add customer ID and actual churn values

customer_risk["customer_id"] = df["customer_id"]

customer_risk["actual_churn"] = df["churn"]


# Create risk categories

def assign_risk(probability):
    if probability >= 0.70:
        return "High Risk"
    elif probability >= 0.30:
        return "Medium Risk"
    else:
        return "Low Risk"


customer_risk["risk_category"] = customer_risk["churn_probability"].apply(assign_risk)


# Display results

customer_risk[
    [
        "customer_id",
        "churn_probability",
        "actual_churn",
        "risk_category"
    ]
].head()

,customer_id,churn_probability,actual_churn,risk_category
0,10001,0.00,0,Low Risk
1,10002,0.00,0,Low Risk
2,10003,0.00,0,Low Risk
3,10004,0.00,0,Low Risk
4,10005,0.01,0,Low Risk


## Customer Risk Distribution

Customers are grouped into risk categories based on their predicted churn probability.

This analysis helps the business understand how many customers require immediate retention attention.

In [29]:
# Count customers by risk category

risk_summary = customer_risk["risk_category"].value_counts().reset_index()


# Rename columns

risk_summary.columns = [
    "Risk Category",
    "Customer Count"
]


# Display results

risk_summary

,Risk Category,Customer Count
0,Low Risk,8496
1,High Risk,1503
2,Medium Risk,1


## Customer Risk Segmentation Summary

Customers were segmented into three risk categories based on predicted churn probability.

Results:

| Risk Category | Customer Count |
|---|---|
| Low Risk | 8,496 |
| Medium Risk | 1 |
| High Risk | 1,503 |

Business interpretation:

The model identified 1,503 customers as high-risk churn candidates.

These customers should be prioritised for retention activities because they represent the greatest potential revenue risk.

The majority of customers were classified as low risk, indicating a stable customer base.

The small number of medium-risk customers suggests that the model is making highly confident predictions, which should be considered when interpreting results.

## Revenue at Risk from High-Risk Customers

The churn prediction model identifies customers who are most likely to leave.

By analysing the revenue contribution of high-risk customers, the business can estimate the financial impact of potential churn and prioritise retention efforts.

In [30]:
# Add monthly revenue to customer risk dataframe

customer_risk["monthly_revenue"] = df["monthly_revenue"]


# Calculate revenue at risk from high-risk customers

high_risk_revenue = customer_risk[
    customer_risk["risk_category"] == "High Risk"
]["monthly_revenue"].sum()


# Count high-risk customers

high_risk_customers = customer_risk[
    customer_risk["risk_category"] == "High Risk"
].shape[0]


# Display results

print("High-risk customers:", high_risk_customers)

print("Revenue at risk: £", high_risk_revenue)

High-risk customers: 1503
Revenue at risk: £ 110337


## Revenue at Risk Analysis

The churn prediction model was used to identify high-risk customers and estimate the potential revenue impact.

Results:

| Metric | Value |
|---|---|
| High-risk customers | 1,503 |
| Revenue at risk | £110,337 |

Business interpretation:

The model identified 1,503 customers who are most likely to churn.

These customers represent £110,337 in potential monthly revenue loss.

By using churn predictions, the business can prioritise retention activities towards customers with the highest financial impact.

Recommended actions:

- Contact high-value high-risk customers first
- Provide personalised retention offers
- Improve engagement among inactive customers
- Address customer experience issues identified through complaints and satisfaction scores

## Machine Learning Final Conclusions

This notebook developed and evaluated multiple machine learning models to predict customer churn for InsightFlow Analytics.

Models developed:

- Logistic Regression
- Random Forest
- Gradient Boosting

Model evaluation showed:

- Logistic Regression achieved strong baseline performance with 95.00% cross-validation accuracy.
- Random Forest achieved the highest validation performance with 99.99% average cross-validation accuracy.
- Gradient Boosting achieved similar performance with 99.98% average cross-validation accuracy.

Random Forest was selected as the final model because it provided the strongest overall performance and also allowed analysis of feature importance.

## Key Churn Prediction Insights

Feature importance analysis identified the main factors associated with customer churn.

The strongest predictive features were:

1. Features used
2. Monthly login activity
3. Last login activity
4. Complaint count
5. Customer satisfaction score

Business interpretation:

Customer engagement and customer experience were the strongest indicators of churn risk.

Customers showing reduced platform usage, lower engagement, increased complaints, or lower satisfaction should receive proactive retention attention.

## Customer Risk and Revenue Impact

The final model generated churn probability scores and classified customers into risk categories.

Risk segmentation results:

| Risk Category | Customers |
|---|---:|
| Low Risk | 8,496 |
| Medium Risk | 1 |
| High Risk | 1,503 |

The high-risk customer group represents:

£110,337 in potential monthly revenue exposure.

Business recommendation:

Retention campaigns should focus on high-risk customers first, prioritising customers with higher revenue contribution and stronger churn signals.

## Final Business Recommendation

Machine learning provides InsightFlow Analytics with a proactive approach to customer retention.

Instead of waiting for customers to leave, the business can identify customers at risk and take preventative action.

Recommended retention strategy:

- Target high-risk customers with personalised engagement campaigns
- Encourage product adoption among customers with low feature usage
- Monitor inactive customers through login behaviour
- Improve customer experience by addressing complaints quickly
- Use churn predictions regularly to support retention decisions

The project demonstrates how data analytics and machine learning can transform customer behaviour data into actionable business decisions.

## Export Customer Risk Predictions

The machine learning model generates a churn probability for each customer.

These predictions are converted into risk categories so that business teams can identify customers requiring retention action.

The output will be exported for use in the Power BI dashboard.

In [31]:
customer_risk.head()

,monthly_revenue,tenure_months,monthly_login_count,average_session_time_minutes,features_used,last_login_days_ago,complaint_count,response_time_hours,resolution_time_hours,customer_satisfaction_score,...,age_group_26-35,age_group_36-45,age_group_46-60,age_group_60+,subscription_type_Enterprise,subscription_type_Professional,churn_probability,customer_id,actual_churn,risk_category
0,79,35,19.666667,28.124427,5.083333,29.666667,0.0,0.000000,0.000000,10.0,...,False,False,False,True,False,True,0.00,10001,0,Low Risk
1,79,24,14.583333,24.804342,5.833333,34.166667,2.0,13.014601,56.085322,9.0,...,False,False,False,False,False,True,0.00,10002,0,Low Risk
2,29,44,14.333333,24.192618,4.916667,30.250000,1.0,4.425014,6.192709,7.0,...,True,False,False,False,False,False,0.00,10003,0,Low Risk
3,29,29,17.083333,22.674429,5.416667,26.583333,0.0,0.000000,0.000000,10.0,...,False,False,False,False,False,False,0.00,10004,0,Low Risk
4,29,35,10.333333,24.052414,4.500000,28.083333,3.0,22.489549,82.144367,9.0,...,False,False,False,True,False,False,0.01,10005,0,Low Risk


## Export Customer Risk Predictions

The customer risk dataframe contains the machine learning prediction results for each customer.

For Power BI, we only need the key business fields:

- Customer ID
- Churn probability
- Actual churn outcome
- Risk category

These fields will allow the dashboard to identify customers requiring retention attention.

In [32]:
customer_risk_predictions = customer_risk[
    [
        "customer_id",
        "churn_probability",
        "actual_churn",
        "risk_category"
    ]
]

customer_risk_predictions.head()

,customer_id,churn_probability,actual_churn,risk_category
0,10001,0.00,0,Low Risk
1,10002,0.00,0,Low Risk
2,10003,0.00,0,Low Risk
3,10004,0.00,0,Low Risk
4,10005,0.01,0,Low Risk


## Save Customer Risk Predictions

The cleaned customer risk table will now be exported as a CSV file.

This file will be imported into Power BI to analyse customer risk distribution and revenue exposure.

In [33]:
customer_risk_predictions.to_csv(
    "customer_risk_predictions.csv",
    index=False
)

print("Customer risk predictions exported successfully.")

Customer risk predictions exported successfully.


In [34]:
import os

os.path.exists("customer_risk_predictions.csv")

True

## Check Feature Importance Data

The machine learning model identified which customer behaviours were most important in predicting churn.

This information will be used in Power BI to create a feature importance visual showing the main churn drivers.

In [35]:
%whos

Variable                     Type                          Data/Info
--------------------------------------------------------------------
GradientBoostingClassifier   ABCMeta                       <class 'sklearn.ensemble.<...>dientBoostingClassifier'>
LogisticRegression           type                          <class 'sklearn.linear_mo<...>stic.LogisticRegression'>
OneHotEncoder                type                          <class 'sklearn.preproces<...>_encoders.OneHotEncoder'>
RandomForestClassifier       ABCMeta                       <class 'sklearn.ensemble.<...>.RandomForestClassifier'>
X                            DataFrame                               region age_grou<...>[10000 rows x 13 columns]
X_encoded                    DataFrame                           monthly_revenue  te<...>[10000 rows x 21 columns]
X_test                       DataFrame                           monthly_revenue  te<...>n[2000 rows x 21 columns]
X_train                      DataFrame                   

## Export Feature Importance Data

The Random Forest model identified the most important factors contributing to customer churn.

This dataset will be imported into Power BI to create a feature importance visual showing the main churn drivers.

The exported file will contain:

- Feature name
- Importance score

In [36]:
feature_importance.head()

,Feature,Importance
4,features_used,0.251289
2,monthly_login_count,0.229130
5,last_login_days_ago,0.183059
6,complaint_count,0.162343
9,customer_satisfaction_score,0.132212


## Save Feature Importance Data

The feature importance results will now be exported as a CSV file.

This file will be used in Power BI to display the factors that have the strongest influence on customer churn predictions.

In [37]:
feature_importance.to_csv(
    "feature_importance.csv",
    index=False
)

print("Feature importance exported successfully.")

Feature importance exported successfully.


## Verify Feature Importance Export

We will check that the CSV file has been created successfully.

In [38]:
import os

os.path.exists("feature_importance.csv")

True

## Export Model Comparison Data

During the machine learning phase, we evaluated three different classification models:

- Logistic Regression
- Random Forest
- Gradient Boosting

The model comparison dataset contains the performance metrics for each model.

This file will allow Power BI to display a comparison of model performance, including accuracy, precision, recall, F1-score, and ROC-AUC.

In [39]:
model_comparison

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.9385,0.858871,0.707641,0.775956,0.973627
1,Random Forest,1.0000,1.000000,1.000000,1.000000,1.000000
2,Gradient Boosting,1.0000,1.000000,1.000000,1.000000,1.000000


## Save Model Comparison Data

The machine learning model evaluation results will now be exported as a CSV file.

This dataset will be used in Power BI to compare model performance and communicate why the Random Forest model was selected for customer risk prediction.

In [40]:
model_comparison.to_csv(
    "model_comparison.csv",
    index=False
)

print("Model comparison exported successfully.")

Model comparison exported successfully.


## Verify Model Comparison Export

We will confirm that the model comparison CSV file has been created successfully.

In [41]:
import os

os.path.exists("model_comparison.csv")

True

## Export Risk Summary Data

The machine learning model classified customers into different risk categories based on their predicted probability of churn.

The risk summary dataset contains the number of customers in each category.

This file will be used in Power BI to create customer risk distribution visuals.

In [42]:
risk_summary

,Risk Category,Customer Count
0,Low Risk,8496
1,High Risk,1503
2,Medium Risk,1


## Save Risk Summary Data

The customer risk summary will now be exported as a CSV file.

This dataset provides a simple summary of customer distribution across risk categories and will be used for executive dashboard reporting.

In [43]:
risk_summary.to_csv(
    "risk_summary.csv",
    index=False
)

print("Risk summary exported successfully.")

Risk summary exported successfully.


## Verify Risk Summary Export

We will confirm that the risk summary CSV file has been created successfully.

In [44]:
import os

os.path.exists("risk_summary.csv")

True

## Verify Power BI Export Files

Before importing data into Power BI, we will confirm that all required CSV files have been created successfully.

This final validation step ensures that the dashboard will be built using complete and consistent datasets.

In [45]:
import os

files = [
    "customer_risk_predictions.csv",
    "feature_importance.csv",
    "model_comparison.csv",
    "risk_summary.csv"
]

for file in files:
    print(file, ":", os.path.exists(file))

customer_risk_predictions.csv : True
feature_importance.csv : True
model_comparison.csv : True
risk_summary.csv : True
